# TerraScout AI — Producto 1: ETL Geoespacial y EDA Real

Notebook técnico de ingesta, limpieza y análisis exploratorio alineado con [`actors/apify_etl/my_actor/etl_core.py`](../actors/apify_etl/my_actor/etl_core.py) y [`actors/apify_eda/my_actor/eda_core.py`](../actors/apify_eda/my_actor/eda_core.py). Compatible con **Google Colab** (clona el repositorio automáticamente) y **Jupyter local**.

> **Nota para Google Colab:** si el repositorio en GitHub está público, esta notebook lo clona automáticamente en `/content/Capstone`. Si está privado, ejecutá primero `git clone` con tu token o montá la carpeta desde Google Drive.

## 1. Objetivo y Política de Datos Reales

- **Cero datos sintéticos (`ZERO DUMMY DATA`):** `100%` fuentes reales en [`data/raw/`](../data/raw/) (`1.200` avisos de Mercado Libre, `288` campings OSM/SoloCampings, `237` perfiles y `1.754` reseñas de Google Maps) y [`data/aux_topografia_cordoba.csv`](../data/aux_topografia_cordoba.csv) (`16` localidades con relieve, NDVI y riesgo ambiental).
- **Reglas de negocio desacopladas:** cotización ARS/USD, rango válido de superficie (`500–20.000 m²`), pendiente máxima (`<= 12°`) y umbral de oportunidad (`20%` bajo mediana zonal) definidos en [`config/business_rules.json`](../config/business_rules.json).
- **Salida compartida:** `dataset_limpio.csv` (`894` lotes únicos) alimenta tanto el **EDA/Dashboard de Producto 1** como las features y el target `log(precio_m2_usd)` del **Modelo Hedónico de Producto 2**.

## 2. Ejecución del ETL Geoespacial y Embudo de Depuración

La función `run_etl_pipeline()` ejecuta:
1. Unificación monetaria (`ARS → USD`) y rescate de superficie (`m²`) ignorando distancias (`km`, `cuadras`, `mts del río`).
2. Filtrado de outliers de precio/m² (IQR) y rangos de superficie aptos para camping/glamping.
3. Deduplicación de publicaciones repetidas y cruce espacial **Haversine** contra campings cercanos, demanda Google Maps y topografía local.
4. Etiquetado de oportunidades P1 (`es_oportunidad_p1`) comparando contra la mediana zonal.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    colab_repo = Path('/content/Capstone')
    if not (colab_repo / 'config' / 'business_rules.json').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/JonathanHiriart/Capstone.git', str(colab_repo)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'pandas', 'numpy', 'scipy', 'scikit-learn'])
    os.chdir(colab_repo)
    REPO_ROOT = colab_repo
else:
    REPO_ROOT = Path('.').resolve()
    if not (REPO_ROOT / 'config' / 'business_rules.json').exists() and (REPO_ROOT.parent / 'config' / 'business_rules.json').exists():
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from actors.apify_etl.my_actor.etl_core import run_etl_pipeline
from actors.apify_eda.my_actor.eda_core import run_eda_pipeline

df_clean, funnel = run_etl_pipeline()
display(pd.DataFrame(funnel))
print(f'Dataset limpio oficial: {len(df_clean)} lotes x {df_clean.shape[1]} columnas | Oportunidades P1: {int(df_clean["es_oportunidad_p1"].sum())}')

## 3. Análisis Exploratorio (EDA), Correlaciones Reales y Voz del Turista

La función `run_eda_pipeline()` genera las 3 hojas oficiales del tablero de Looker Studio (`contenido_graficas_dashboard.xlsx`):
- **Hoja 1:** Resumen agregado por localidad y riesgo ambiental (lotes, mediana `USD/m²`, pendiente media, campings en 5 km).
- **Hoja 2:** Correlaciones estadísticas reales (**Pearson, Spearman, Kendall e Información Mutua**) contra `log(precio_m2_usd)`, que fundamentan la selección de variables del modelo hedónico en [`02_modelado_hedonico.ipynb`](./02_modelado_hedonico.ipynb).
- **Hoja 3:** Ranking de lotes en oportunidad P1 + tópicos NLP extraídos sobre `1.754` reseñas reales de turistas en Google Maps.

In [ ]:
clean_path = REPO_ROOT / 'data' / 'processed' / 'dataset_limpio.csv'
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False, encoding='utf-8')

eda_res = run_eda_pipeline(dataset_limpio_source=str(clean_path))
print('Resumen EDA:', json.dumps(eda_res['summary'], indent=2, ensure_ascii=False))

print('\n--- Hoja 1: Resumen por Localidad y Riesgo ---')
display(eda_res['csv_1_localidades_y_riesgo'].head(10))

print('\n--- Hoja 2: Correlaciones Reales con log(precio_m2_usd) ---')
df_c2 = eda_res['csv_2_correlaciones_y_distribuciones']
display(df_c2[df_c2['bloque'] == 'correlacion_features'])

print('\n--- Hoja 3: Muestra de Tópicos NLP (Voz del Turista) ---')
df_c3 = eda_res['csv_3_oportunidades_y_voz_turista']
display(df_c3[df_c3['seccion'] != 'oportunidad_lote'].head(10))